In [2]:
import pandas as pd
import numpy as np

# Step 1: Load the dataset
df = pd.read_csv("marketing_summary.csv")

In [3]:
# Step 2: Convert date columns to datetime
df['date'] = pd.to_datetime(df['date'])
df['report_generated'] = pd.to_datetime(df['report_generated'])

In [10]:
# Step 3: Keep only the official report fields
df = df[['date', 'users_active', 'total_sales', 'new_customers']]

In [11]:
# Step 4: Fill missing values
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])
    elif df[col].dtype in ['float64', 'int64']:
        df[col] = df[col].fillna(df[col].median())


In [12]:
# Step 5: Cap numeric outliers using IQR method
def cap_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    return np.clip(series, Q1 - 1.5*IQR, Q3 + 1.5*IQR)

num_cols = df.select_dtypes(include=['float64', 'int64']).columns
for col in num_cols:
    df[col] = cap_outliers(df[col])

In [13]:
# Step 6: Round all float columns to 2 decimals
df[num_cols] = df[num_cols].round(2)

In [14]:
# Step 7: Export final cleaned file
df.to_csv("final_cleaned_marketing_summary.csv", index=False)